In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
from scipy.stats import zscore

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
samples = sorted(list(np.loadtxt(f'{projdir}/txt/allc_table/ips_cluster_donor.txt', dtype=str)))

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
def process_bed(s):
    bins = pd.read_csv(f'{projdir}/bed/bins/ips_cluster_donor/{s}.chrom100k_CGN-Both.sparse.bed.gz', sep='\t', header=None)
    bins.index = [f'{x}_{y}' for x,y in zip(bins[0], bins[1])]
    return bins

with ProcessPoolExecutor() as executor:
    bins_results = list(tqdm(executor.map(process_bed, samples), total=len(samples)))

In [ ]:
bins_results[0]

In [ ]:
bins_mcg = [x[4] / x[5] for x in bins_results]
bins_df = pd.concat(bins_mcg, axis=1)
bins_df.columns = samples
bins_cov = pd.concat([x[5] for x in bins_results], axis=1)
bins_cov.columns = samples
bins_df.shape

In [ ]:
bins_df.head()

In [ ]:
donors = ['C29', 'C37', 'C38', 'C39']

In [ ]:
ips_samples = [f'{x}_IPS' for x in donors]
indir = '/u/project/cluo/terencew/igvf/50_line/snmCT/pseudobulk_allc'
def process_bed(s):
    pmds = pd.read_csv(f'{indir}/bed/bins/donor_time/{s}.chrom100k_CGN-Both.sparse.bed.gz', sep='\t', header=None)
    pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
    return pmds

with ProcessPoolExecutor() as executor:
    ips_results = list(tqdm(executor.map(process_bed, ips_samples), total=len(ips_samples)))

In [ ]:
mature_mcg = [x[4] / x[5] for x in ips_results]
mature_df = pd.concat(mature_mcg, axis=1)
mature_df.columns = ips_samples
mature_df = mature_df[~mature_df.index.duplicated()]
mature_cov = pd.concat([x[5] for x in ips_results], axis=1)
mature_cov.columns = ips_samples
mature_cov = mature_cov[~mature_cov.index.duplicated()]
mature_df.shape

In [ ]:
# keep bins with >=10x coverage in every profile, no imputation
cov = pd.concat([bins_cov, mature_cov], axis=1).fillna(0)
keep = cov.index[(cov >= 10).all(axis=1)]
bins_df = bins_df.loc[keep]
final_mature = mature_df.loc[keep]
final_mature.shape

In [ ]:
merged = pd.concat([bins_df, final_mature], axis=1)
merged.shape

In [ ]:
merged.head()

In [ ]:
s = 'C29'
main_diff = np.abs(merged[f'IPS_main_{s}'] - merged[f'{s}_IPS'])
alt_diff = np.abs(merged[f'IPS_alt_{s}'] - merged[f'{s}_IPS'])

sns.histplot(main_diff,  color="tab:blue")
sns.histplot(alt_diff,  color="tab:orange")
plt.show()

In [ ]:
main_diff.mean(), alt_diff.mean(), \
main_diff.var(), alt_diff.var()

In [ ]:
s = 'C39'
main_diff = np.abs(merged[f'IPS_main_{s}'] - merged[f'{s}_IPS'])
alt_diff = np.abs(merged[f'IPS_alt_{s}'] - merged[f'{s}_IPS'])

sns.histplot(main_diff,  color="tab:blue")
sns.histplot(alt_diff,  color="tab:orange")
plt.show()

In [ ]:
main_diff.mean(), alt_diff.mean(), \
main_diff.var(), alt_diff.var()

In [ ]:
s = 'C37'
main_diff = np.abs(merged[f'IPS_main_{s}'] - merged[f'{s}_IPS'])
alt_diff = np.abs(merged[f'IPS_alt_{s}'] - merged[f'{s}_IPS'])

sns.histplot(main_diff,  color="tab:blue")
sns.histplot(alt_diff,  color="tab:orange")
plt.show()

In [ ]:
main_diff.mean(), alt_diff.mean(), \
main_diff.var(), alt_diff.var()

In [ ]:
corr_df = merged.corr()
corr_df

In [ ]:
sns.heatmap(corr_df, cmap='coolwarm')

In [ ]:
sns.clustermap(merged, cmap='coolwarm')

### get better corr df

In [ ]:
centered = merged - merged.mean(axis=1).values[:, None]
centered.shape

In [ ]:
# corr_df

In [ ]:
z = merged.sub(merged.mean(axis=1), axis=0)\
          .div(merged.std(axis=1).replace(0, 1), axis=0)

corr_df = z.corr()
corr_df.head()

In [ ]:
sns.heatmap(corr_df, cmap='coolwarm', vmin=-0.5, vmax=0.5)

### write out

In [ ]:
corr_df = merged.corr()
sns.heatmap(corr_df, cmap='coolwarm')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5/'
corr_df.to_csv(f'{outdir}/ips_subtype_mcg_mature_corr.csv', sep='\t')

In [ ]:
!date